<a href="https://colab.research.google.com/github/Srushanth/RAG/blob/advanced-rag-stage-1/Advanced-RAG/notebooks/advanced-rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Advanced RAG Experiments with LlamaIndex

This notebook walks through **three Advanced RAG techniques** as separate experiments:

| # | Technique | Stage | Description |
|---|-----------|-------|-------------|
| 1 | **HyDE** | Pre-retrieval | Generates a hypothetical answer → embeds that for retrieval |
| 2 | **Re-ranking** | Post-retrieval | Cross-encoder rescores retrieved chunks |
| 3 | **Sub-Question** | Pre-retrieval | Decomposes complex queries into sub-questions |

We compare each against a **Baseline** (naive vector retrieval).

---
## 1. Setup & Configuration

In [1]:
! pip install "llama-index-core>=0.14.21" "llama-index-embeddings-huggingface>=0.7.0" "llama-index-llms-google-genai>=0.9.1" "llama-index-postprocessor-sbert-rerank>=0.4.0" "sentence-transformers>=4.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16,

In [1]:
import os
import nest_asyncio

nest_asyncio.apply()

In [44]:
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex
from llama_index.core.tools import ToolMetadata
from llama_index.core.tools import QueryEngineTool
from llama_index.core import SimpleDirectoryReader
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.query_engine import TransformQueryEngine
from llama_index.core.question_gen import LLMQuestionGenerator
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.postprocessor.sbert_rerank import SentenceTransformerRerank
from llama_index.core.indices.query.query_transform import HyDEQueryTransform

In [5]:
# Set your Gemini API Key
os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"

In [6]:
Settings.llm = GoogleGenAI(model="gemini-3-flash-preview")

In [7]:
f"LLM: {Settings.llm.model}"

'LLM: gemini-3-flash-preview'

In [8]:
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", device="cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
f"Embed model: {Settings.embed_model.model_name}"

'Embed model: BAAI/bge-small-en-v1.5'

---
## 2. Load Documents & Build Index

In [10]:
documents = SimpleDirectoryReader("data").load_data()

In [11]:
f"Loaded {len(documents)} document(s)"

'Loaded 1 document(s)'

In [12]:
index = VectorStoreIndex.from_documents(documents=documents, show_progress=True)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/640 [00:00<?, ?it/s]

In [13]:
baseline_engine = index.as_query_engine(similarity_top_k=5)

In [14]:
TEST_QUERY = "What are the main topics discussed in the docment?"

---
## 3. Baseline — Naive Vector Retrieval

Simple vector similarity search with no enhancements. This serves as our control experiment.

In [15]:
baseline_response = baseline_engine.query(TEST_QUERY)

In [17]:
print(baseline_response.response)

The document focuses on the following main topics:

*   **Environmental Progress:** A comprehensive report on environmental initiatives and progress for the year 2025.
*   **Sustainability Reporting and Assurance:** Detailed assurance statements and templates used for verifying sustainability reports.
*   **Fiscal Year Review Statements:** Specific review documentation for the 2024 fiscal year (FY24), including a CCF Review Statement.
*   **Performance Verification:** Formal assurance and review processes conducted by third-party users to validate environmental and sustainability data.


In [26]:
print(f"Source nodes: {len(baseline_response.source_nodes)}")
for i, node in enumerate(baseline_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    print(f"  [{i}] score={score:.4f} | {node.text[:100]}...")

Source nodes: 5
  [1] score=0.5499 | docx</rdf:li>
            </rdf:Alt>
         </dc:title>
      </rdf:Description>
   </rdf:RDF>
</x...
  [2] score=0.5497 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22...
  [3] score=0.5497 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22...
  [4] score=0.5497 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22...
  [5] score=0.5482 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22...


---
## 4. Experiment 1: HyDE (Hypothetical Document Embeddings)

### How it works
1. The LLM generates a **hypothetical answer** to the query (even though it doesn't have access to the documents yet).
2. That hypothetical answer is **embedded** and used for vector search instead of the raw query.
3. Since the hypothetical answer is more semantically similar to actual document passages than a short query, retrieval quality improves.

### When to use
- Queries that are short or abstract (e.g., "What is RAG?")
- When there's a vocabulary mismatch between queries and documents

In [20]:
hyde_transform = HyDEQueryTransform(include_original=True)

In [21]:
hyde_query_bundle = hyde_transform(TEST_QUERY)

In [22]:
print("=" * 80)
print("HyDE — HYPOTHETICAL DOCUMENT GENERATED")
print("=" * 80)
print(f"Original query: {TEST_QUERY}")
print(f"\nHypothetical document(s):")
for i, emb_str in enumerate(hyde_query_bundle.embedding_strs):
    print(f"\n--- Hypothetical Doc {i+1} ---")
    print(emb_str[:500])

HyDE — HYPOTHETICAL DOCUMENT GENERATED
Original query: What are the main topics discussed in the docment?

Hypothetical document(s):

--- Hypothetical Doc 1 ---
It appears that you haven't provided the document text yet! However, to show you the level of detail and structure I can provide, **here is a sample passage answering that question based on a hypothetical report about Artificial Intelligence in Healthcare:**

### Sample Answer:
"The document provides a comprehensive analysis of the integration of **Artificial Intelligence (AI) within the healthcare sector**, focusing on three primary areas: diagnostic advancements, operational efficiency, and et

--- Hypothetical Doc 2 ---
What are the main topics discussed in the docment?


In [23]:
hyde_engine = TransformQueryEngine(query_engine=baseline_engine, query_transform=hyde_transform)

In [24]:
hyde_response = hyde_engine.query(TEST_QUERY)

In [25]:
hyde_response.response

'The document focuses on environmental progress and reporting for the year 2025. It details specific goal-related progress, likely through the use of charts and data tracking to illustrate environmental performance and achievements.'

In [27]:
print(f"Source nodes: {len(hyde_response.source_nodes)}")
for i, node in enumerate(hyde_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    print(f"  [{i}] score={score:.4f} | {node.text[:100]}...")

Source nodes: 5
  [1] score=0.6646 | @Oj[poQDs]|ҠFXH=蝎0;On<fV F9cb<Os!qLpqަK(Op[S8#"X3Sg
ilym$(Qڋp(b͵w{Se#q,elhܸRDz>...
1_=7#!VoJ$Gqn># Q$$.@E7r&)
S tcو...
  [3] score=0.6586 | wi@
UjQS"YhM\@ue42#yTM,5uKL䁴5^@'OAa20*1jMe4<YU ")Ĭ[E...
<</Count 6/Kids[967 0 R 972 0 R 975 0 R 1011 0 R 1014 0 R 1017 0 R]/Parent 23557 0 R/Type/Page...
<</D(20024...


---
## 5. Experiment 2: Re-ranking (Cross-Encoder)

### How it works
1. Vector search retrieves **top-K** candidate chunks (fast but approximate).
2. A **cross-encoder** model scores each `(query, chunk)` pair jointly — this is much more accurate than bi-encoder similarity.
3. Chunks are re-sorted by the cross-encoder score, and the **top-N** are kept.

### Model
We use `BAAI/bge-reranker-v2-m3` — a free, local cross-encoder. No API key needed.

### When to use
- When initial retrieval returns partially relevant results
- When precision matters more than recall

In [29]:
reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-v2-m3", device="cuda", top_n=3)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [30]:
rerank_engine = index.as_query_engine(similarity_top_k=5, node_postprocessors=[reranker])

In [34]:
rerank_response = rerank_engine.query(TEST_QUERY)

In [35]:
rerank_response.response

'The main topics discussed in the document include environmental progress and sustainability reporting. Specifically, the material covers an assurance statement for sustainability reports and a review statement for the fiscal year 2024 related to CCF.'

In [36]:
print(f"Source nodes (after re-ranking): {len(rerank_response.source_nodes)}")
for i, node in enumerate(rerank_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    print(f"  [{i}] score={score:.4f} | {node.text[:100]}...")

Source nodes (after re-ranking): 3
  [1] score=0.0283 | docx</rdf:li>
            </rdf:Alt>
         </dc:title>
      </rdf:Description>
   </rdf:RDF>
</x...
  [2] score=0.0233 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22...
  [3] score=0.0233 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22...


In [37]:
# Compare scores: before vs after re-ranking
print("=" * 80)
print("SCORE COMPARISON: Baseline vs Re-ranked")
print("=" * 80)

print("\nBaseline (vector similarity scores):")
for i, node in enumerate(baseline_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    print(f"  [{i}] {score:.4f} | {node.text[:80]}...")

print("\nRe-ranked (cross-encoder scores):")
for i, node in enumerate(rerank_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    print(f"  [{i}] {score:.4f} | {node.text[:80]}...")

SCORE COMPARISON: Baseline vs Re-ranked

Baseline (vector similarity scores):
  [1] 0.5499 | docx</rdf:li>
            </rdf:Alt>
         </dc:title>
      </rdf:Descriptio...
  [2] 0.5497 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://w...
  [3] 0.5497 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://w...
  [4] 0.5497 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://w...
  [5] 0.5482 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://w...

Re-ranked (cross-encoder scores):
  [1] 0.0283 | docx</rdf:li>
            </rdf:Alt>
         </dc:title>
      </rdf:Descriptio...
  [2] 0.0233 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://w...
  [3] 0.0233 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://w...


---
## 6. Experiment 3: Sub-Question Query Engine

### How it works
1. The LLM analyzes the user's question and **decomposes** it into independent sub-questions.
2. Each sub-question is sent to the appropriate query engine tool.
3. All sub-answers are **synthesized** into a single coherent response.

### When to use
- Complex, multi-part questions (e.g., "Compare X and Y", "What are the pros and cons of Z?")
- When a single retrieval pass wouldn't cover all aspects

In [41]:
query_engine_tools = [
    QueryEngineTool(
        query_engine=baseline_engine,
        metadata=ToolMetadata(
            name="Document search",
            description=(
                "Provides information from the loaded documents. "
                "Use this tool to answer questions about the document content."
            )
        )
    )
]

In [45]:
question_gen = LLMQuestionGenerator.from_defaults()

In [46]:
sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=query_engine_tools,
    question_gen=question_gen,
)

In [47]:
COMPLEX_QUERY = "What are the main topics discussed and how do they relate to each other?"

In [48]:
sub_q_response = sub_question_engine.query(COMPLEX_QUERY)

Generated 2 sub questions.
[Document search] Q: What are the main topics discussed in the documents?
[Document search] Q: How do the main topics discussed in the documents relate to each other?
[Document search] A: The documents primarily discuss environmental progress and sustainability reporting for the year 2025. Key topics include the assurance of sustainability reports, environmental performance tracking, and the formal statements verifying these results.
[Document search] A: The documentation of environmental progress is directly integrated with sustainability reporting and the verification of that data through assurance statements. Specifically, the tracking of environmental performance serves as the core content for sustainability reports, which are then subject to formal assurance processes to validate the accuracy of the information provided. These elements relate to each other by forming a structured cycle of reporting, where environmental achievements are documented accordi

In [49]:
sub_q_response.response

'The main topics center on environmental progress and sustainability reporting for the year 2025. This includes the tracking of environmental performance, the development of sustainability reports, and the formal assurance and verification of the resulting data.\n\nThese topics are interconnected through a structured cycle of reporting and validation. Environmental performance tracking provides the core data used to populate sustainability reports. These reports are then subjected to formal assurance processes, where verification statements are issued to confirm the accuracy and integrity of the documented environmental achievements.'

In [50]:
print(f"Source nodes: {len(sub_q_response.source_nodes)}")
for i, node in enumerate(sub_q_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    score_str = f"score={score:.4f}" if score is not None else "score=N/A"
    print(f"  [{i}] {score_str} | {node.text[:100]}...")

Source nodes: 12
  [1] score=N/A | Sub question: What are the main topics discussed in the documents?
Response: The documents primarily...
  [2] score=N/A | Sub question: How do the main topics discussed in the documents relate to each other?
Response: The ...
  [3] score=0.5701 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22...
  [4] score=0.5701 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22...
  [5] score=0.5701 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22...
  [6] score=0.5699 | 1801 1102 1401 1801 1102 1401 1801 1102 1401 1801 1102 1401 1801 1102 1401 1801 1102 1401 1801 1102 ...
  [7] score=0.5693 | wi@
UjQS"YhM\@ue42#yTM,5uKL䁴5^@'OAa20*1jMe4<YU ")Ĭ[E...
  [8] score=0.5606 | 1-c002 79.b7c64cc, 2024/07/16-07:59:40        ">
   <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22...
  [9] score=0.5606 | 1-c002 

---
## 7. Comparison: All Techniques Side-by-Side

Let's compare all four approaches on the same query.

In [51]:
COMPARISON_QUERY = "What are the key concepts explained in these documents?"

results = {}

print("Running Baseline...")
results["Baseline"] = baseline_engine.query(COMPARISON_QUERY)

print("Running HyDE...")
results["HyDE"] = hyde_engine.query(COMPARISON_QUERY)

print("Running Re-ranking...")
results["Re-ranking"] = rerank_engine.query(COMPARISON_QUERY)

print("Running Sub-Question...")
results["Sub-Question"] = sub_question_engine.query(COMPARISON_QUERY)

print("Done!")

Running Baseline...
Running HyDE...
Running Re-ranking...
Running Sub-Question...
Generated 1 sub questions.
[Document search] Q: What are the key concepts explained in the documents?
[Document search] A: The documents focus on environmental progress and sustainability reporting for 2025. Key concepts include the assurance of sustainability reports and the use of template assurance statements for such reporting.
Done!


In [52]:
# Print side-by-side comparison
for technique, response in results.items():
    print("=" * 80)
    print(f"  {technique.upper()}")
    print("=" * 80)
    print(response.response)
    num_nodes = len(response.source_nodes) if hasattr(response, 'source_nodes') else 0
    print(f"\n  [Source nodes: {num_nodes}]")
    print()

  BASELINE
Key concepts include the assurance of sustainability reports and the implementation of a template assurance statement, specifically a medium-level version. These topics are associated with documentation regarding environmental progress reporting for 2025.

  [Source nodes: 5]

  HYDE
The documents focus on environmental progress reporting and the formal assurance process for sustainability data. Key concepts include:

*   **Environmental Progress Tracking:** Documentation of initiatives and advancements related to environmental goals for the year 2025.
*   **Sustainability Report Assurance:** The process of providing third-party verification and assurance statements for reported sustainability information.
*   **Assurance Frameworks:** The use of specific templates and standards for creating assurance statements, including "medium" level assurance for sustainability reports.
*   **External Verification:** The involvement of specialized organizations and users in auditing and

---
## Summary

| Technique | Best For | Trade-off |
|-----------|----------|----------|
| **Baseline** | Simple, well-formed queries | Fast but limited |
| **HyDE** | Short/abstract queries, vocabulary mismatch | Extra LLM call per query |
| **Re-ranking** | Improving precision from noisy retrieval | Extra model inference (cross-encoder) |
| **Sub-Question** | Complex multi-part questions | Multiple LLM calls, slower |